# Parse FIX
Resolve structured message rows against the dictionary, and route them.

In [ ]:
project_root = "."
source = "text.messages"
start = None
end = None
fix_dictionary = "data/fix"
rules = None
protocols = None
fields = None
catalog = "rekep"
catalog_properties = {}
target_pattern = "fixmessage.{category}"
instrument_source = "market.instruments"
instrument_snapshot_every = 3_600_000_000_000
static_values = {}
merge_by = True
batch_row_size = 65_536
commit_row_size = 250_000
limit = None

In [ ]:
from pathlib import Path

import pyarrow
import pyarrow.compute as pc
from pyiceberg.expressions import And, GreaterThanOrEqual, LessThan, LessThanOrEqual

from rekep.fix.registry import FixRegistry
from rekep.fix.fields import FieldRules
from rekep.fix.rules import Rules
from rekep.fix.transcribe import FixCodec
from rekep.iceberg import IcebergDataset
from rekep.market import Instrument
from rekep.market.event import DAY
from rekep.text import FixMessage, FixMessageRules
from rekep.times import unix_of
from rekep.urls import Url


def _local_path(value):
    parsed = Url.from_string(str(value))
    if parsed.scheme not in {"", "file", "local"}:
        return str(value)
    path = Path(parsed.path)
    return str(path if path.is_absolute() else Path(project_root).resolve() / path)


def _window(lower, upper, column="unix"):
    predicates = []
    if lower is not None:
        predicates.append(GreaterThanOrEqual(column, lower))
    if upper is not None:
        predicates.append(LessThan(column, upper))
    return None if not predicates else predicates[0] if len(predicates) == 1 else And(*predicates)


event_rules = FixMessageRules() if rules is None else FixMessageRules.from_dict(rules)
protocol_rules = Rules() if protocols is None else Rules.from_dict(protocols)
registry = FixRegistry(cache_dir=_local_path(fix_dictionary), offline=True, announce=print)
field_rules = FieldRules() if fields is None else FieldRules.from_dict(fields)
codec = FixCodec(rules=protocol_rules, registry=registry, fields=field_rules)
field = FixMessage.into_field()
if static_values:
    from rekep.text.text_file import parsed_field_of, static_columns_of

    field = parsed_field_of(field, static_columns_of(static_values))
messages = IcebergDataset(
    name=source, catalog=catalog, properties=dict(catalog_properties)
)


def _without_message(batch):
    """A market row's raw line, dropped: `kwargs` carries everything it held.

    An all-null column run-length and dictionary encodes to nothing on disk,
    which is what makes one stored shape across the three tables affordable.
    """
    at = batch.schema.get_field_index("message")
    columns = list(batch.columns)
    columns[at] = pyarrow.nulls(batch.num_rows, batch.schema.field(at).type)
    return pyarrow.RecordBatch.from_arrays(columns, schema=batch.schema)

In [ ]:
targets = {}
buffers = {}
held_rows = {}
read = written = skipped = 0
observed_lower = observed_upper = None

# The window is read off the *stored* recording clock, because that is what
# the message stage partitioned on. `unix` moves when a transaction time
# resolves, so filtering on it here would drop rows the interval owns.
lower, upper = unix_of(start), unix_of(end, upper=True)


def _target(category):
    target = targets.get(category)
    if target is None:
        target = targets[category] = IcebergDataset(
            name=target_pattern.format(category=category),
            catalog=catalog,
            properties=dict(catalog_properties),
            field=field,
            commit_row_size=commit_row_size,
            sort_by=("unix", "msg_seq_num", "hash"),
        )
    return target


def _flush(category):
    global written, skipped
    batches = buffers.pop(category, [])
    count = held_rows.pop(category, 0)
    if not count:
        return
    landed = _target(category).append_arrow_table(
        pyarrow.Table.from_batches(batches), merge_by=merge_by
    )
    written += landed
    skipped += count - landed


for staged in messages.read_arrow_reader(field, row_filter=_window(lower, upper)):
    if limit is not None and read + staged.num_rows > limit:
        staged = staged.slice(0, max(0, limit - read))
    if not staged.num_rows:
        continue
    read += staged.num_rows
    # Resolve, and never re-read `message`: the protocol and its version are
    # stored columns, so nothing here categorises or re-splits a line.
    batch = FixMessage.resolve_arrow_batch(staged, codec)
    bounds = pc.min_max(batch.column("unix")).as_py()
    observed_lower = bounds["min"] if observed_lower is None else min(observed_lower, bounds["min"])
    observed_upper = (
        bounds["max"] + 1 if observed_upper is None else max(observed_upper, bounds["max"] + 1)
    )
    categories = protocol_rules.into_arrow_category_array(
        batch.column("protocol_code"), batch.column("etype")
    )
    names = sorted(pc.unique(categories).to_pylist())
    for category in names:
        part = batch if len(names) == 1 else batch.filter(pc.equal(categories, category))
        # `message` is the content of record only where the row could not be
        # used as a FIX message; on a market row `kwargs` carries everything
        # it held, and an all-null column costs nothing on disk.
        if category == "market":
            part = _without_message(part)
        buffers.setdefault(category, []).append(part)
        held_rows[category] = held_rows.get(category, 0) + part.num_rows
        if commit_row_size and held_rows[category] >= commit_row_size:
            _flush(category)
    if limit is not None and read >= limit:
        break
for category in list(buffers):
    _flush(category)

In [ ]:
lower = lower if lower is not None else observed_lower
upper = upper if upper is not None else observed_upper
instrument_table = IcebergDataset(
    name=instrument_source, catalog=catalog, properties=dict(catalog_properties)
)


def _instrument_seeds():
    if lower is None:
        return []
    recent = And(GreaterThanOrEqual("unix", lower - DAY), LessThanOrEqual("unix", lower))
    reader = instrument_table.read_arrow_reader(
        Instrument.into_field(), row_filter=recent, order_by=("unix", "version", "hash")
    )
    latest = {}
    for batch in reader:
        for row in batch.to_pylist():
            instrument = Instrument.from_dict(row)
            current = latest.get(instrument.xhash)
            if current is None or (instrument.unix, instrument.version, instrument.hash) > (
                current.unix,
                current.version,
                current.hash,
            ):
                latest[instrument.xhash] = instrument
    return sorted(latest.values(), key=lambda row: (row.unix, row.xhash))


def _market_messages():
    reader = _target("market").read_arrow_reader(
        field, row_filter=_window(lower, upper), order_by=("unix", "msg_seq_num", "hash")
    )
    for batch in reader:
        for row in batch.to_pylist():
            message = FixMessage.from_dict(row)
            if not message.is_instrument_version:
                yield message


def _instrument_flush(pending):
    if not pending:
        return 0
    table = pyarrow.Table.from_pylist(
        [{**one.into_fixmessage().into_dict(), **static_values} for one in pending],
        schema=field.into_arrow_schema(),
    )
    pending.clear()
    return _target("market").append_arrow_table(table, merge_by=merge_by)


instrument_versions = instrument_written = 0
instrument_held = []
if lower is not None and upper is not None:
    versions = Instrument.from_fixmessages(
        _market_messages(),
        registry=registry,
        instruments=_instrument_seeds(),
        snapshot_every=instrument_snapshot_every,
        snapshot_until=upper,
    )
    for instrument in versions:
        if instrument.unix < lower or instrument.unix >= upper:
            continue
        instrument_held.append(instrument)
        instrument_versions += 1
        if commit_row_size and len(instrument_held) >= commit_row_size:
            instrument_written += _instrument_flush(instrument_held)
    instrument_written += _instrument_flush(instrument_held)

result = {
    "read": read,
    "written": written + instrument_written,
    "skipped": skipped + instrument_versions - instrument_written,
    "raw_written": written,
    "instrument_versions": instrument_versions,
    "instrument_written": instrument_written,
    "targets": {category: target.name for category, target in targets.items()},
}
result